# EMG Signal Processing in Google Colab

This notebook is a lightweight Google Colab example for the EMG tutorial. It installs the required packages, loads a synthetic EMG signal, and demonstrates preprocessing and feature extraction.

In [ ]:
!pip install numpy pandas scipy matplotlib seaborn scikit-learn >/dev/null

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal

sns.set_theme(style='whitegrid')

In [ ]:
# Create a simple synthetic EMG signal
fs = 1000
t = np.arange(0, 4, 1/fs)
emg = 0.2*np.sin(2*np.pi*0.4*t) + 0.4*np.exp(-((t-1.2)/0.15)**2) + 0.3*np.exp(-((t-2.7)/0.12)**2) + 0.08*np.random.randn(len(t))

df = pd.DataFrame({'time_s': t, 'emg_uV': emg})
df.head()

In [ ]:
# Preprocessing steps
def remove_dc_offset(x):
    return x - np.mean(x)

def bandpass_filter(x, fs):
    b, a = signal.butter(4, [20/(0.5*fs), 450/(0.5*fs)], btype='band')
    return signal.filtfilt(b, a, x)

def notch_filter(x, fs, freq=50):
    b, a = signal.iirnotch(freq, 30, fs)
    return signal.sosfiltfilt(signal.tf2sos(b, a), x)

x_dc = remove_dc_offset(df['emg_uV'].to_numpy())
x_bp = bandpass_filter(x_dc, fs)
x_notch = notch_filter(x_bp, fs, freq=50)
x_rect = np.abs(x_notch)

plt.figure(figsize=(12, 4))
plt.plot(df['time_s'], df['emg_uV'], label='Raw EMG')
plt.plot(df['time_s'], x_rect, label='Rectified EMG', linewidth=1.5)
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.title('EMG preprocessing example')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Simple features
mav = np.mean(np.abs(x_rect))
rms = np.sqrt(np.mean(x_rect**2))
peak = np.max(np.abs(x_rect))

features = pd.DataFrame({
    'feature': ['MAV', 'RMS', 'Peak amplitude'],
    'value': [mav, rms, peak]
})
features